# Modelo Bi-LSTM

In [1]:
import pandas as pd
df = pd.read_csv('df_exportado.csv')

## Labels e split

In [2]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold, train_test_split
from sklearn.feature_extraction.text import CountVectorizer
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"A usar: {device}")

label_map = {'Human':0,'OpenAI':1,'Google':2,'Meta':3,'Anthropic':4}
y = df['Label'].map(label_map).values
num_classes = 5

train_texts, test_texts, y_train, y_test = train_test_split(
    df['Text'], y, test_size=0.2, random_state=42, stratify=y
)

A usar: cuda


## Vetorização

In [3]:
from collections import Counter
import re

print("A criar vocabulário e a processar sequências...")

# Parâmetros para as sequências
max_words = 100000  # Tamanho do vocabulário
max_seq_length = 150  # Tamanho máximo de cada texto (em palavras)

# 1. Função simples de tokenização
def tokenize(text):
    # Limpa um bocado o texto e separa por palavras
    return re.findall(r'\b\w+\b', str(text).lower())

# 2. Criar o vocabulário com as palavras mais comuns do treino
todas_as_palavras = [palavra for texto in train_texts for palavra in tokenize(texto)]
contagem = Counter(todas_as_palavras)

# Dicionário de palavra -> ID (0 é reservado para Padding, 1 para Unknown/Desconhecido)
vocab = {palavra: i+2 for i, (palavra, _) in enumerate(contagem.most_common(max_words - 2))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

# 3. Função para converter texto em sequência de IDs com Padding
def encode_and_pad(text):
    tokens = tokenize(text)
    # Converte para IDs (se não existir no vocab, usa 1)
    seq = [vocab.get(word, 1) for word in tokens]
    
    # Truncar (cortar) se for maior que o limite
    if len(seq) > max_seq_length:
        seq = seq[:max_seq_length]
    # Padding (preencher com 0s) se for menor que o limite
    else:
        seq = seq + [0] * (max_seq_length - len(seq))
    return seq

# 4. Transformar os dados de treino
X_train_seq = [encode_and_pad(t) for t in train_texts]
y_train_np = np.array(y_train)

# Transformar para tensores PyTorch
# IMPORTANTE: Para o nn.Embedding da LSTM, os inputs têm de ser do tipo torch.long (inteiros)
X_train_tensor = torch.tensor(X_train_seq, dtype=torch.long)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.long)

print(f"Shape do X_train_tensor: {X_train_tensor.shape}") # Deve ser (num_exemplos, max_seq_length)

A criar vocabulário e a processar sequências...
Shape do X_train_tensor: torch.Size([149657, 150])


## Modelo

In [10]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, num_layers=2, dropout=0.3):
        super().__init__()
        
        # 1. Camada Embedding: Transforma o ID da palavra num vetor denso (ex: 128 dimensões)
        # padding_idx=0 diz ao modelo para ignorar os zeros que adicionamos
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 2. A Camada Bi-LSTM
        # batch_first=True porque os nossos tensores têm o batch size na primeira dimensão (batch, seq_len)
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            bidirectional=True, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # 3. Camadas Densas para Classificação
        # Multiplicamos hidden_dim por 2 porque a LSTM é bidirecional (junta a leitura da esquerda->direita com direita->esquerda)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        # x shape: (batch_size, seq_length)
        
        # Passa pelo embedding
        embedded = self.embedding(x) 
        # embedded shape: (batch_size, seq_length, embedding_dim)
        
        # Passa pela LSTM
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # hidden contém o último estado oculto. Sendo bidirecional e com 2 camadas, 
        # queremos juntar as saídas da última camada que leu "para a frente" e "para trás".
        # hidden shape: (num_layers * num_directions, batch_size, hidden_dim)
        hidden_forward = hidden[-2, :, :]  # Última camada forward
        hidden_backward = hidden[-1, :, :] # Última camada backward
        
        # Concatenar os dois estados
        hidden_cat = torch.cat((hidden_forward, hidden_backward), dim=1)
        # hidden_cat shape: (batch_size, hidden_dim * 2)
        
        # Passar pelas camadas densas finais
        out = self.fc1(hidden_cat)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out

## Funções de Treino e Avaliação

In [11]:
def train_model(model, loader, criterion, optimizer):
    model.train()
    
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        # 1. Acumular a loss do batch
        total_loss += loss.item() * X_batch.size(0)
        
        # 2. Descobrir qual a classe prevista (o índice com maior probabilidade)
        _, preds = torch.max(outputs, dim=1)
        
        # 3. Contar quantas previsões bateram certo com o y_batch
        correct_predictions += torch.sum(preds == y_batch).item()
        total_samples += y_batch.size(0)
        
    # Calcular a média final
    epoch_loss = total_loss / total_samples
    epoch_acc = correct_predictions / total_samples
    
    return epoch_loss, epoch_acc

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

## Grid Search

In [12]:
from sklearn.utils.class_weight import compute_class_weight

param_grid = {
    'learning_rate': [0.001, 0.01], 
    'batch_size': [32, 64],
    'epochs': [5]
}

keys, values = zip(*param_grid.items())
configs = [dict(zip(keys, v)) for v in itertools.product(*values)]

# OTIMIZAÇÃO: Usar apenas 10.000 textos para descobrir os melhores parâmetros (LSTMs são mais lentas)
tamanho_amostra = min(10000, len(X_train_tensor))
indices_amostra = torch.randperm(len(X_train_tensor))[:tamanho_amostra]
X_grid = X_train_tensor[indices_amostra]
y_grid = y_train_tensor[indices_amostra]

kf = KFold(n_splits=3, shuffle=True, random_state=42)

melhor_acc = 0
melhores_params = None

# Calcular pesos automaticamente com base na proporção das classes no conjunto de treino
pesos_calculados = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Converter para tensor do PyTorch
pesos = torch.tensor(pesos_calculados, dtype=torch.float32).to(device)

print(f"Pesos calculados automaticamente: {pesos}")

# Hiperparâmetros da arquitetura LSTM
EMBEDDING_DIM = 100
HIDDEN_DIM = 128

print("\nGrid Search Rápido...")
for config in configs:
    fold_acc = []
    for train_idx, val_idx in kf.split(X_grid.numpy()):
        
        train_dataset = TensorDataset(X_grid[train_idx], y_grid[train_idx])
        val_dataset = TensorDataset(X_grid[val_idx], y_grid[val_idx])

        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config['batch_size'])

        model = BiLSTM(max_words, EMBEDDING_DIM, HIDDEN_DIM, num_classes).to(device)
        criterion = nn.CrossEntropyLoss(weight=pesos)
        optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

        for epoch in range(config['epochs']):
            train_model(model, train_loader, criterion, optimizer)

        acc = evaluate(model, val_loader)
        fold_acc.append(acc)

    media = np.mean(fold_acc)
    print(f"{config} -> Accuracy: {media:.4f}")

    if media > melhor_acc:
        melhor_acc = media
        melhores_params = config

print("\nMelhores parâmetros:", melhores_params)

Pesos calculados automaticamente: tensor([0.6176, 0.7360, 1.1290, 1.1293, 3.9871], device='cuda:0')

Grid Search Rápido...
{'learning_rate': 0.001, 'batch_size': 32, 'epochs': 5} -> Accuracy: 0.8198
{'learning_rate': 0.001, 'batch_size': 64, 'epochs': 5} -> Accuracy: 0.8211
{'learning_rate': 0.01, 'batch_size': 32, 'epochs': 5} -> Accuracy: 0.7848
{'learning_rate': 0.01, 'batch_size': 64, 'epochs': 5} -> Accuracy: 0.7933

Melhores parâmetros: {'learning_rate': 0.001, 'batch_size': 64, 'epochs': 5}


## Treino Final

In [13]:
from sklearn.utils.class_weight import compute_class_weight

X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(

    X_train_tensor.numpy(), y_train_tensor.numpy(), test_size=0.15, stratify=y_train_tensor.numpy(), random_state=42

)

# Calcular pesos automaticamente com base na proporção das classes no conjunto de treino
pesos_calculados = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_final),
    y=y_train_final
)

# Converter para tensor do PyTorch
pesos = torch.tensor(pesos_calculados, dtype=torch.float32).to(device)
EMBEDDING_DIM = 100
HIDDEN_DIM = 128

# 1. Enviar os dados diretamente para a GPU logo na criação
X_train_final = torch.tensor(X_train_final, dtype=torch.long).to(device)
y_train_final = torch.tensor(y_train_final, dtype=torch.long).to(device)
X_val_final = torch.tensor(X_val_final, dtype=torch.long).to(device)
y_val_final = torch.tensor(y_val_final, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_final, y_train_final)
val_dataset = TensorDataset(X_val_final, y_val_final)

# 2. Se por acaso os dados não couberem todos na GPU e der erro de "Out of Memory", 
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

# Inicializar o modelo
model_final = BiLSTM(max_words, EMBEDDING_DIM, HIDDEN_DIM, num_classes).to(device)

criterion = nn.CrossEntropyLoss(weight=pesos.to(device)) 
optimizer = torch.optim.Adam(model_final.parameters(), lr=0.001)

print("\nTreino final...")
for epoch in range(15):
    train_loss, train_acc = train_model(model_final, train_loader, criterion, optimizer)
    
    val_acc = evaluate(model_final, val_loader)
    
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print("\nTreino Concluído!")


Treino final...
Epoch 01 | Train Loss: 0.4527 | Train Acc: 0.8087 | Val Acc: 0.9295
Epoch 02 | Train Loss: 0.1411 | Train Acc: 0.9481 | Val Acc: 0.9618
Epoch 03 | Train Loss: 0.0653 | Train Acc: 0.9773 | Val Acc: 0.9791
Epoch 04 | Train Loss: 0.0339 | Train Acc: 0.9884 | Val Acc: 0.9796
Epoch 05 | Train Loss: 0.0208 | Train Acc: 0.9927 | Val Acc: 0.9827
Epoch 06 | Train Loss: 0.0150 | Train Acc: 0.9950 | Val Acc: 0.9850
Epoch 07 | Train Loss: 0.0106 | Train Acc: 0.9966 | Val Acc: 0.9855
Epoch 08 | Train Loss: 0.0072 | Train Acc: 0.9975 | Val Acc: 0.9837
Epoch 09 | Train Loss: 0.0078 | Train Acc: 0.9973 | Val Acc: 0.9836
Epoch 10 | Train Loss: 0.0068 | Train Acc: 0.9978 | Val Acc: 0.9871
Epoch 11 | Train Loss: 0.0046 | Train Acc: 0.9984 | Val Acc: 0.9850
Epoch 12 | Train Loss: 0.0040 | Train Acc: 0.9987 | Val Acc: 0.9859
Epoch 13 | Train Loss: 0.0035 | Train Acc: 0.9989 | Val Acc: 0.9864
Epoch 14 | Train Loss: 0.0044 | Train Acc: 0.9987 | Val Acc: 0.9861
Epoch 15 | Train Loss: 0.0023 |